!pip install music21 pretty_midi pyfluidsynth
!apt-get install -y fluidsynth

In [ ]:
# Extracting Notes from a Bach Score Using music21

from music21 import corpus

bach = corpus.getComposer('bach')[0]
score = corpus.parse(bach)
notes = [str(n.pitch) for n in score.flatten().notes if n.isNote]
print(notes[:20])

['F4', 'F4', 'C4', 'A3', 'F3', 'G4', 'C5', 'C4', 'G3', 'E3', 'C4', 'F4', 'A4', 'C4', 'A3', 'F3', 'F3', 'A3', 'F4', 'D4']


In [ ]:
# Collecting Notes and Chords from 30 Bach Pieces

from music21 import corpus

all_notes = []
paths = corpus.getComposer('bach')[:30]

for path in paths:
    score = corpus.parse(path)
    chords = score.chordify()   # merges all voices into one sequence of notes/chords
    for element in chords.flatten().notes:
        # each element becomes text like "C4" or "C4.E4.G4"
        all_notes.append('.'.join(p.nameWithOctave for p in element.pitches))

print("Total notes/chords:", len(all_notes))
print("Unique ones:", len(set(all_notes)))
print(all_notes[:10])

Total notes/chords: 2575
Unique ones: 1375
['F3.A3.C4.F4', 'E3.G3.C4.G4.C5', 'E3.G3.C4.C5', 'F3.A3.C4.F4.A4', 'F3.A3.C4.A4', 'D3.A3.D4.F4', 'D3.F3.A3.D4.F4', 'A2.A3.F4.C5', 'A2.A3.C4.F4.C5', 'B-2.F3.F4.B-4.D5']


In [ ]:
# Converting Notes into Number Sequences for Training the Music Model

import numpy as np
from tensorflow.keras.utils import to_categorical

sequence_length = 50

pitch_names = sorted(set(all_notes))
n_vocab = len(pitch_names)
note_to_int = {n: i for i, n in enumerate(pitch_names)}

network_input = []
network_output = []

for i in range(len(all_notes) - sequence_length):
    seq_in = all_notes[i : i + sequence_length]
    seq_out = all_notes[i + sequence_length]
    network_input.append([note_to_int[n] for n in seq_in])
    network_output.append(note_to_int[seq_out])

n_patterns = len(network_input)

X = np.reshape(network_input, (n_patterns, sequence_length, 1)) / float(n_vocab)
y = to_categorical(network_output, num_classes=n_vocab)

print("Training examples:", n_patterns)
print("Vocabulary size:", n_vocab)
print("X shape:", X.shape)
print("y shape:", y.shape)

Training examples: 2525
Vocabulary size: 1375
X shape: (2525, 50, 1)
y shape: (2525, 1375)


In [ ]:
# Building the LSTM Neural Network Model for Music Generation

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

model = Sequential([
    Input(shape=(X.shape[1], X.shape[2])),
    LSTM(256, return_sequences=True),
    Dropout(0.3),
    LSTM(256),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(n_vocab, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1375)           │       353,375 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,208,671 (4.61 MB)

 Trainable params: 1,208,671 (4.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X, y, epochs=30, batch_size=64)

model.save('music_model.keras')

Epoch 1/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 7.2098
Epoch 2/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 7.0377
Epoch 3/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.9716
Epoch 4/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.9387
Epoch 5/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.9266
Epoch 6/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.9222
Epoch 7/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.9168
Epoch 8/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 6.9171
Epoch 9/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 6.9138
Epoch 10/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 6.9122
Epoch 11/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 6.9083
Epoch 12/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.9030
Epoch 13/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.8947
Epoch 14/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.8670
Epoch 15/30
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 6.8041
Epoc

In [ ]:
model.fit(X, y, epochs=70, batch_size=64)
model.save('music_model.keras')

Epoch 1/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 4.0006
Epoch 2/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 3.7307
Epoch 3/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 3.5174
Epoch 4/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 3.3271
Epoch 5/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 3.0759
Epoch 6/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 2.8002
Epoch 7/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 2.6641
Epoch 8/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 2.4612
Epoch 9/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 2.2865
Epoch 10/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 2.1142
Epoch 11/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 1.9046
Epoch 12/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 1.7887
Epoch 13/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 1.6515
Epoch 14/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 1.5285
Epoch 15/70
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.4227
Epoc

In [ ]:
# Generating New Music with the Model and Saving It as a MIDI File

import random
from music21 import note, chord, stream

int_to_note = {i: n for n, i in note_to_int.items()}

def sample(probs, temperature=1.0):
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs) / np.sum(np.exp(probs))
    return np.random.choice(len(probs), p=probs)

# pick a random starting seed from the training data
start = random.randint(0, len(network_input) - 1)
pattern = list(network_input[start])
generated = []

for _ in range(200):                       # generate 200 notes/chords
    x = np.reshape(pattern, (1, sequence_length, 1)) / float(n_vocab)
    probs = model.predict(x, verbose=0)[0]
    idx = sample(probs, temperature=0.8)
    generated.append(int_to_note[idx])
    pattern.append(idx)
    pattern = pattern[1:]

# convert to a MIDI file
output = []
offset = 0
for item in generated:
    if '.' in item:                        # it's a chord
        c = chord.Chord(item.split('.'))
        c.offset = offset
        output.append(c)
    else:                                  # it's a single note
        n = note.Note(item)
        n.offset = offset
        output.append(n)
    offset += 0.5

stream.Stream(output).write('midi', fp='generated_music.mid')
print("Saved generated_music.mid")

Saved generated_music.mid


In [ ]:
# Playing the Generated MIDI Music as Audio in the Notebook

import pretty_midi
from IPython.display import Audio

pm = pretty_midi.PrettyMIDI('generated_music.mid')
audio = pm.fluidsynth(fs=44100)
Audio(audio, rate=44100)

In [ ]:
# Downloading the Generated Music and the Trained Model from Colab

from google.colab import files
files.download('generated_music.mid')
files.download('music_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Converting the Generated MIDI Music into a WAV Audio File

import pretty_midi
from scipy.io import wavfile
import numpy as np

pm = pretty_midi.PrettyMIDI('generated_music.mid')
audio = pm.fluidsynth(fs=44100)
wavfile.write('generated_music.wav', 44100, (audio * 32767).astype(np.int16))

In [ ]:
# Saving and Downloading the Note-to-Number Dictionary as a JSON File

import json
from google.colab import files

with open('note_to_int.json', 'w') as f:
    json.dump(note_to_int, f)

files.download('note_to_int.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Converting the Generated Music to WAV and Downloading It

import pretty_midi
import numpy as np
from scipy.io import wavfile
from google.colab import files

pm = pretty_midi.PrettyMIDI('generated_music.mid')
audio = pm.fluidsynth(fs=44100)
wavfile.write('generated_music.wav', 44100, (audio * 32767).astype(np.int16))
files.download('generated_music.wav')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>